# AmberTools User Guide

**Version:** AmberTools25 (released April 30, 2025)

**Audience:** Computational chemists and molecular simulation users running biomolecular workflows on HPC systems

## TL;DR

This guide explains **how to use AmberTools effectively on this HPC system**.

**What is AmberTools?** AmberTools is a suite of biomolecular simulation and analysis programs that can be used independently or together for complete molecular dynamics workflows.

Run one simple case first:

```bash
# 1. Load the module
module load ambertools/25

# 2. Verify install (example tool from the suite)
cpptraj --version

# 3. Run a minimal command
antechamber -h
```

It emphasizes:
- Quick setup for first-time users
- Common AmberTools task patterns in shared HPC environments
- SLURM-ready CPU and GPU production workflows

The goal is to **get you from zero to your first result in 15 minutes**, then move into production workflows.

## Table of Contents

- [TL;DR](#tldr)
- [PART I — Getting Started (Read Once)](#part-i--getting-started-read-once)
  - [TL;DR: 5-Minute QuickStart](#tldr-5-minute-quickstart)
  - [Step 1: Load the Module](#step-1-load-the-module)
  - [Step 2: Verify Setup](#step-2-verify-setup)
  - [Step 3: Run Your First Job](#step-3-run-your-first-job)
- [PART II — How to Use AmberTools (Read Carefully Once)](#part-ii--how-to-use-ambertools-read-carefully-once)
  - [What You Need to Know](#what-you-need-to-know)
  - [Core Executables and Common Options](#core-executables-and-common-options)
  - [Resource Requests](#resource-requests)
- [PART III — Workflows (Use As Needed)](#part-iii--workflows-use-as-needed)
  - [Workflow 1: CPU MD Job on a Build Partition](#workflow-1-cpu-md-job-on-a-build-partition)
  - [Workflow 2: GPU MD Job on a GPU Partition](#workflow-2-gpu-md-job-on-a-gpu-partition)
  - [Workflow 3: Trajectory Analysis with cpptraj](#workflow-3-trajectory-analysis-with-cpptraj)
- [PART IV — Troubleshooting (Skim When Broken)](#part-iv--troubleshooting-skim-when-broken)
  - [Symptom-Based Lookup](#symptom-based-lookup)
  - [FAQ](#faq)
  - [External Resources](#external-resources)

---

## PART I — Getting Started (Read Once)

**Purpose:** "I just want this to work."

This part should feel **quick and confidence-building**. By the end, you will have a working setup and have run your first command successfully.

### TL;DR: 5-Minute QuickStart

If you just want to run a command:

```bash
# 1. Load the module
module load ambertools/25

# 2. Verify it worked
cpptraj --version

# 3. Run a basic command
antechamber -h
```

**Expected result:** You should see AmberTools command output and usage text.

---

### Step 1: Load the Module

All software on this system is managed through **environment modules (Lmod)**. Start by checking available Amber/AmberTools module names and versions:

In [ ]:
%%bash
module avail amber 2>&1 | head -30

**Expected output:**
```
...
amber/24.1
ambertools/25
...
```

Now load the module (adjust the exact name to your site listing if needed):

In [ ]:
%%bash
module load ambertools/25
module list 2>&1 | head -20

**Expected output:**
```
Currently Loaded Modules:
  1) ambertools/25
```

---

### Step 2: Verify Setup

Confirm the environment is ready using a commonly used executable from the AmberTools suite:

In [ ]:
%%bash
module load ambertools/25
cpptraj --version 2>&1 | head -10

**Expected output:**
```
CPPTRAJ: Trajectory Analysis. V6.x.x ...
```

✅ **Success!** Your AmberTools environment is ready to use.

---

### Step 3: Run Your First Job

Run a minimal command to validate command access and inspect available options:

In [ ]:
%%bash
module load ambertools/25
antechamber -h 2>&1 | head -40

**Expected output:**
```
Usage: antechamber ...
Options include:
  -i  input file
  -fi input format
  -o  output file
  -fo output format
  -c  charge method
  -nc net charge
...
```

🎉 **You are set up!** Continue to Part II for usage guidance, or jump to Part III for production workflows.

**Next Steps:**
- Part II → understand tools, options, and resource choices.
- Part III → run CPU/GPU SLURM workflows.
- Part IV → troubleshoot common failures quickly.

---

## PART II — How to Use AmberTools (Read Carefully Once)

**Purpose:** "Help me not screw this up."

### What You Need to Know

AmberTools25 major components include: antechamber, MCPB.py, tleap, parmed, sqm, Quick, pbsa, 3D-RISM, sander, gem.pmemd, mdgx, cpptraj, pytraj, and mmpbsa.py.

Typical HPC flow:
1. Build topology/coordinates (tleap, antechamber)
2. Run simulation (sander or pmemd family)
3. Analyze trajectories (cpptraj, pytraj, mmpbsa.py)

#### Supported Environments

| Environment | Status | Notes |
|---|---|---|
| Login node | ✅ For small prep/analysis | Use for setup and short tests only |
| CPU partition (build) | ✅ Recommended for CPU MD | Use `sander.MPI` for parallel runs |
| GPU partition | ✅ Recommended for acceleration | Use `pmemd.cuda` where available |

### Core Executables and Common Options

| Tool | Purpose | Common options/flags |
|---|---|---|
| `antechamber` | Ligand parameter prep | `-i`, `-fi`, `-o`, `-fo`, `-c`, `-nc` |
| `tleap` | Build prmtop/inpcrd | `-f <input.leap>` |
| `sander` | CPU MD engine | `-O`, `-i`, `-o`, `-p`, `-c`, `-r`, `-x` |
| `pmemd.cuda` | GPU MD engine | `-O`, `-i`, `-o`, `-p`, `-c`, `-r`, `-x` |
| `cpptraj` | Trajectory analysis | `-i <analysis.in>` |

To inspect options directly in your environment:

In [ ]:
%%bash
module load ambertools/25
for tool in antechamber tleap sander cpptraj; do
  echo "===== ${tool} help ====="
  ${tool} -h 2>&1 | head -25
  echo
done

**Expected output:**
```
===== antechamber help =====
Usage: antechamber ...
...
===== tleap help =====
Usage: tleap ...
...
```

### Resource Requests

- **Preparation and quick checks:** No allocation needed; run on login node.
- **CPU production MD:** Submit to the build/CPU partition with explicit core and memory requests.
- **GPU production MD:** Submit to a GPU partition and request GPUs explicitly.

⚠️ For interactive debugging on compute resources, use:
```bash
srun --pty -p build -c 8 --mem=32G bash
```

---

## PART III — Workflows (Use As Needed)

**Purpose:** "Show me how to actually do science."

Each workflow is self-contained and uses module-based setup.

### Workflow 1: CPU MD Job on a Build Partition

Create and submit a CPU batch script (example uses `sander.MPI`):

In [ ]:
%%bash
cat > amber_cpu_build.sbatch << 'EOF'
#!/bin/bash
#SBATCH -J amber_cpu_build
#SBATCH -p build
#SBATCH -N 1
#SBATCH -n 16
#SBATCH --mem=64G
#SBATCH -t 04:00:00
#SBATCH -o amber_cpu_%j.out
#SBATCH -e amber_cpu_%j.err

module load ambertools/25

# Example minimization input
sander.MPI -O \
  -i min.in \
  -o min.out \
  -p system.prmtop \
  -c system.inpcrd \
  -r min.rst7 \
  -x min.nc
EOF

sed -n '1,40p' amber_cpu_build.sbatch

**Expected output:**
```
#!/bin/bash
#SBATCH -J amber_cpu_build
#SBATCH -p build
...
module load ambertools/25
sander.MPI -O -i min.in ...
```

Submit and monitor:

In [ ]:
%%bash
sbatch amber_cpu_build.sbatch
squeue -u "$USER" | head -20

**Expected output:**
```
Submitted batch job 123456
JOBID PARTITION NAME           USER ST TIME NODES NODELIST(REASON)
123456 build     amber_cpu... user R  0:03 1     ...
```

### Workflow 2: GPU MD Job on a GPU Partition

Create and submit a GPU batch script (example uses `pmemd.cuda`):

In [ ]:
%%bash
cat > amber_gpu.sbatch << 'EOF'
#!/bin/bash
#SBATCH -J amber_gpu
#SBATCH -p gpu
#SBATCH -N 1
#SBATCH --gres=gpu:1
#SBATCH -c 8
#SBATCH --mem=48G
#SBATCH -t 08:00:00
#SBATCH -o amber_gpu_%j.out
#SBATCH -e amber_gpu_%j.err

module load ambertools/25

# Example MD step on GPU
pmemd.cuda -O \
  -i md.in \
  -o md.out \
  -p system.prmtop \
  -c equil.rst7 \
  -r md.rst7 \
  -x md.nc
EOF

sed -n '1,40p' amber_gpu.sbatch

**Expected output:**
```
#!/bin/bash
#SBATCH -p gpu
#SBATCH --gres=gpu:1
...
module load ambertools/25
pmemd.cuda -O -i md.in ...
```

Submit and monitor:

In [ ]:
%%bash
sbatch amber_gpu.sbatch
squeue -u "$USER" | head -20

**Expected output:**
```
Submitted batch job 123457
JOBID PARTITION NAME       USER ST TIME NODES NODELIST(REASON)
123457 gpu       amber_gpu  user R  0:04 1     ...
```

### Workflow 3: Trajectory Analysis with cpptraj

Run a post-processing workflow after simulation completes:

In [ ]:
%%bash
module load ambertools/25
cat > rmsd.in << 'EOF'
parm system.prmtop
trajin md.nc
rms first :1-200@CA out rmsd_backbone.dat
run
EOF

cpptraj -i rmsd.in 2>&1 | head -40

**Expected output:**
```
CPPTRAJ: Trajectory Analysis...
Reading 'system.prmtop' as Amber Topology
Reading 'md.nc' as Amber NetCDF
RMSD to first frame ...
```

---

## PART IV — Troubleshooting (Skim When Broken)

**Purpose:** "Something failed. Give me answers fast."

### Symptom-Based Lookup

| Symptom | Likely Cause | Fix |
|---|---|---|
| `command not found` for `cpptraj`/`sander` | Module not loaded | `module load ambertools/25` then retry |
| `ERROR: Could not open file` | Missing input files in submit directory | Check file paths and `pwd`; use absolute paths if needed |
| SLURM job stays pending | Partition/resource mismatch | Check `squeue -u $USER` and reduce requested resources |
| `CUDA driver` or GPU runtime error | Submitted to CPU partition or wrong GPU env | Submit to GPU partition and verify GPU allocation (`nvidia-smi`) |
| Job killed due to memory | Under-requested RAM | Increase `#SBATCH --mem` and rerun |

### FAQ

**Q: Do I need a SLURM allocation for every AmberTools command?**

A: No. Small prep/inspection commands can run on login nodes. Use SLURM for production or long-running jobs.

**Q: Which executable should I use for GPU MD?**

A: Use `pmemd.cuda` in a GPU partition. For CPU parallel runs, use `sander.MPI` in a CPU/build partition.

**Q: How do I check job progress?**

A: Use `squeue -u $USER` while running, and inspect `*.out` / `*.err` files after completion.

### External Resources

- Official AmberTools page: https://ambermd.org/AmberTools.php
- Download and release page: https://ambermd.org/GetAmber.php#ambertools
- Amber 2025 Reference Manual (syntax/details): https://ambermd.org/doc12/Amber25.pdf
- Official tutorials: https://ambermd.org/tutorials/
- GPU support notes: https://ambermd.org/GPUSupport.php